# AcidNet Qwen3.5-4B Unsloth T4 Colab Path

This notebook is the T4-oriented AcidNet training path derived from the official Unsloth free-Colab example style.

It differs from the main AcidNet Colab notebook in two ways:
- it follows the official Unsloth install pinning more closely for free Google Colab T4 environments
- it defaults to a short smoke run instead of trying a full 50k/4k run on a slow non-bf16 GPU

Important:
- Training uses the Hugging Face model path, not the deployment GGUF.
- The preferred T4 try path here is `16bit LoRA` (`load_in_4bit=False`), mirroring the official notebook style.
- `TRAIN_MODE=smoke` is the safe default. Move to `full` only after the smoke run looks healthy.
- In Colab, prefer a `Secrets` entry named `HF_TOKEN`; the notebook falls back to the normal `HF_TOKEN` environment variable only if the secret is absent.


In [ ]:
%%capture
import os, importlib.util

%pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil = f"pillow=={PIL.__version__}"
    except Exception:
        _numpy = "numpy"
        _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0 datasets accelerate peft huggingface_hub sentencepiece protobuf
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import os

WORKDIR = Path("/content/acidnet_t4")
DATA_ROOT = WORKDIR / "data"
PROMPT_PACK_DIR = DATA_ROOT / "prompt_packs"
SFT_DIR = DATA_ROOT / "sft"
PREFERENCES_DIR = DATA_ROOT / "preferences"
TRAINING_DIR = DATA_ROOT / "training"
EVAL_DIR = DATA_ROOT / "eval"
GGUF_DIR = DATA_ROOT / "gguf"

for path in [WORKDIR, PROMPT_PACK_DIR, SFT_DIR, PREFERENCES_DIR, TRAINING_DIR, EVAL_DIR, GGUF_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def resolve_hf_token() -> tuple[str, str]:
    try:
        from google.colab import userdata  # type: ignore
        secret_token = userdata.get("HF_TOKEN")
        if secret_token:
            return secret_token, "colab_secret"
    except Exception:
        pass
    env_token = os.environ.get("HF_TOKEN", "")
    if env_token:
        return env_token, "environment"
    return "", "missing"

HF_TOKEN, HF_TOKEN_SOURCE = resolve_hf_token()
DATASET_REPO = os.environ.get("ACIDNET_DATASET_REPO", "acidsound/acidnet_dataset")
MODEL_REPO = os.environ.get("ACIDNET_MODEL_REPO", "acidsound/acidnet_model")
DATASET_RUN = os.environ.get(
    "ACIDNET_DATASET_RUN",
    "qwen3_5_4b_runtime_dialogue_unsloth_wsl_full_headsync_20260308",
)
OUTPUT_RUN = os.environ.get(
    "ACIDNET_OUTPUT_RUN",
    f"qwen3_5_4b_runtime_dialogue_t4_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}",
)
TRAINING_MODEL = os.environ.get("ACIDNET_TRAINING_MODEL", "unsloth/Qwen3.5-4B")
BASE_MODEL_METADATA = os.environ.get("ACIDNET_BASE_MODEL", "Qwen/Qwen3.5-4B")

TRAIN_MODE = os.environ.get("ACIDNET_TRAIN_MODE", "smoke").strip().lower()
LOAD_IN_4BIT = False
MAX_SEQ_LENGTH = 1024
LEARNING_RATE = 2e-4
SMOKE_BATCH_SIZE = 2
SMOKE_GRAD_ACCUM = 4
FULL_BATCH_SIZE = 2
FULL_GRAD_ACCUM = 8
NUM_TRAIN_EPOCHS = 1
WARMUP_RATIO = 0.05
SMOKE_WARMUP_STEPS = 5
SMOKE_MAX_STEPS = 30
EVAL_STEPS = 1000
SAVE_STEPS = 1000
LORA_RANK = 16
LORA_ALPHA = 16

UPLOAD_ADAPTER_TO_HF = False
EXPORT_GGUF = False

ADAPTER_DIR = TRAINING_DIR / f"{OUTPUT_RUN}_adapter"
LOCAL_RUN_SPEC_PATH = TRAINING_DIR / f"{OUTPUT_RUN}_run_spec.json"
LOCAL_MANIFEST_PATH = TRAINING_DIR / f"{OUTPUT_RUN}_colab_manifest.json"
GGUF_EXPORT_DIR = GGUF_DIR / OUTPUT_RUN

print({
    "hf_token_source": HF_TOKEN_SOURCE,
    "dataset_repo": DATASET_REPO,
    "model_repo": MODEL_REPO,
    "dataset_run": DATASET_RUN,
    "output_run": OUTPUT_RUN,
    "training_model": TRAINING_MODEL,
    "base_model_metadata": BASE_MODEL_METADATA,
    "train_mode": TRAIN_MODE,
    "load_in_4bit": LOAD_IN_4BIT,
})


In [ ]:
from huggingface_hub import login, notebook_login

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(f"Logged into Hugging Face with HF_TOKEN from {HF_TOKEN_SOURCE}.")
else:
    print("HF_TOKEN is empty. Add a Colab secret named HF_TOKEN or run notebook_login() below.")
    # notebook_login()


In [ ]:
from huggingface_hub import hf_hub_download
import shutil

required_dataset_files = {
    f"runs/{DATASET_RUN}/sft/train.jsonl": SFT_DIR / "train.jsonl",
    f"runs/{DATASET_RUN}/sft/eval.jsonl": SFT_DIR / "eval.jsonl",
}

optional_dataset_files = {
    f"runs/{DATASET_RUN}/sft/bench_train_1024.jsonl": SFT_DIR / "bench_train_1024.jsonl",
    f"runs/{DATASET_RUN}/sft/bench_eval_128.jsonl": SFT_DIR / "bench_eval_128.jsonl",
    f"runs/{DATASET_RUN}/prompt_packs/requests.parquet": PROMPT_PACK_DIR / "requests.parquet",
    f"runs/{DATASET_RUN}/prompt_packs/teacher_outputs.jsonl": PROMPT_PACK_DIR / "teacher_outputs.jsonl",
    f"runs/{DATASET_RUN}/preferences/preferences.parquet": PREFERENCES_DIR / "preferences.parquet",
    f"runs/{DATASET_RUN}/preferences/manifest.json": PREFERENCES_DIR / "manifest.json",
    f"runs/{DATASET_RUN}/manifests/pipeline.json": TRAINING_DIR / "published_pipeline.json",
    f"runs/{DATASET_RUN}/manifests/run_spec.json": TRAINING_DIR / "published_run_spec.json",
    f"runs/{DATASET_RUN}/manifests/gate_report.json": EVAL_DIR / "published_gate_report.json",
    f"runs/{DATASET_RUN}/manifests/publish_manifest.json": TRAINING_DIR / "published_hf_manifest.json",
}

def restore_hf_file(remote_path: str, local_path: Path, *, required: bool) -> Path | None:
    try:
        cached = hf_hub_download(
            repo_id=DATASET_REPO,
            repo_type="dataset",
            filename=remote_path,
            token=HF_TOKEN or None,
        )
    except Exception as exc:
        if required:
            raise
        print(f"skip optional {remote_path}: {exc}")
        return None
    local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(cached, local_path)
    print(f"restored {remote_path} -> {local_path}")
    return local_path

for remote_path, local_path in required_dataset_files.items():
    restore_hf_file(remote_path, local_path, required=True)

for remote_path, local_path in optional_dataset_files.items():
    restore_hf_file(remote_path, local_path, required=False)


In [ ]:
import json
import math
from datasets import load_dataset
import torch

published_run_spec_path = TRAINING_DIR / "published_run_spec.json"
if published_run_spec_path.exists():
    published_spec = json.loads(published_run_spec_path.read_text(encoding="utf-8"))
    MAX_SEQ_LENGTH = int(published_spec.get("max_seq_length", MAX_SEQ_LENGTH))
    LEARNING_RATE = float(published_spec.get("learning_rate", LEARNING_RATE))
    NUM_TRAIN_EPOCHS = int(published_spec.get("num_train_epochs", NUM_TRAIN_EPOCHS))
    EVAL_STEPS = int(published_spec.get("eval_steps", EVAL_STEPS))
    SAVE_STEPS = int(published_spec.get("save_steps", SAVE_STEPS))
    LORA_RANK = int(published_spec.get("lora_rank", LORA_RANK))
    LORA_ALPHA = int(published_spec.get("lora_alpha", LORA_ALPHA))
    WARMUP_RATIO = float(published_spec.get("warmup_ratio", WARMUP_RATIO))

def ensure_smoke_jsonl(full_path: Path, smoke_path: Path, limit: int) -> Path:
    if smoke_path.exists():
        return smoke_path
    dataset = load_dataset("json", data_files=str(full_path), split="train")
    rows = [dataset[index] for index in range(min(limit, len(dataset)))]
    smoke_path.parent.mkdir(parents=True, exist_ok=True)
    with smoke_path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    return smoke_path

if TRAIN_MODE not in {"smoke", "full"}:
    raise ValueError("ACIDNET_TRAIN_MODE must be smoke or full for this T4 notebook.")

SMOKE_TRAIN_PATH = ensure_smoke_jsonl(SFT_DIR / "train.jsonl", SFT_DIR / "bench_train_1024.jsonl", 1024)
SMOKE_EVAL_PATH = ensure_smoke_jsonl(SFT_DIR / "eval.jsonl", SFT_DIR / "bench_eval_128.jsonl", 128)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

if TRAIN_MODE == "smoke":
    SELECTED_TRAIN_PATH = SMOKE_TRAIN_PATH
    SELECTED_EVAL_PATH = SMOKE_EVAL_PATH
    TRAIN_BATCH_SIZE = SMOKE_BATCH_SIZE
    TRAIN_GRAD_ACCUM = SMOKE_GRAD_ACCUM
    SELECTED_MAX_STEPS = SMOKE_MAX_STEPS
    WARMUP_STEPS = SMOKE_WARMUP_STEPS
else:
    SELECTED_TRAIN_PATH = SFT_DIR / "train.jsonl"
    SELECTED_EVAL_PATH = SFT_DIR / "eval.jsonl"
    TRAIN_BATCH_SIZE = FULL_BATCH_SIZE
    TRAIN_GRAD_ACCUM = FULL_GRAD_ACCUM
    SELECTED_MAX_STEPS = None
    full_train_rows = len(load_dataset("json", data_files=str(SELECTED_TRAIN_PATH), split="train"))
    effective_batch_size = TRAIN_BATCH_SIZE * TRAIN_GRAD_ACCUM
    steps_per_epoch = max(1, math.ceil(full_train_rows / effective_batch_size))
    WARMUP_STEPS = max(1, int(steps_per_epoch * NUM_TRAIN_EPOCHS * WARMUP_RATIO))

print({
    "gpu_name": gpu_name,
    "bf16_supported": bf16_supported,
    "training_model": TRAINING_MODEL,
    "train_mode": TRAIN_MODE,
    "selected_train_path": str(SELECTED_TRAIN_PATH),
    "selected_eval_path": str(SELECTED_EVAL_PATH),
    "batch_size": TRAIN_BATCH_SIZE,
    "grad_accum": TRAIN_GRAD_ACCUM,
    "selected_max_steps": SELECTED_MAX_STEPS,
    "warmup_steps": WARMUP_STEPS,
})


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=TRAINING_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    use_gradient_checkpointing="unsloth",
    full_finetuning=False,
)

print({
    "training_model": TRAINING_MODEL,
    "load_in_4bit": LOAD_IN_4BIT,
    "max_seq_length": MAX_SEQ_LENGTH,
})


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


In [ ]:
from datasets import load_dataset
import json

train_raw = load_dataset("json", data_files=str(SELECTED_TRAIN_PATH), split="train")
eval_raw = load_dataset("json", data_files=str(SELECTED_EVAL_PATH), split="train")

def format_record(record):
    if record.get("text"):
        return {"text": record["text"]}
    messages = record["messages"]
    if isinstance(messages, str):
        messages = json.loads(messages)
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
                enable_thinking=False,
            )
            return {"text": text}
        except TypeError:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            return {"text": text}
    return {"text": "\n".join(f"{item['role']}: {item['content']}" for item in messages)}

train_dataset = train_raw.map(lambda record: format_record(record), remove_columns=train_raw.column_names)
eval_dataset = eval_raw.map(lambda record: format_record(record), remove_columns=eval_raw.column_names)

print({
    "train_rows": len(train_dataset),
    "eval_rows": len(eval_dataset),
    "preview": train_dataset[0]["text"][:400],
})


In [ ]:
base_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

def tokenize_batch(batch):
    return base_tokenizer(
        text=batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

tokenized_train_dataset = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_dataset.column_names,
)
tokenized_eval_dataset = eval_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=eval_dataset.column_names,
)

print({
    "base_tokenizer_class": type(base_tokenizer).__name__,
    "tokenized_example": tokenized_train_dataset[0],
})


In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

trainer_kwargs = {
    "output_dir": str(ADAPTER_DIR),
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": TRAIN_GRAD_ACCUM,
    "warmup_steps": WARMUP_STEPS,
    "logging_steps": 1 if TRAIN_MODE == "smoke" else 10,
    "optim": "adamw_8bit",
    "weight_decay": 0.001,
    "lr_scheduler_type": "linear",
    "seed": 3407,
    "report_to": "none",
    "save_strategy": "steps",
    "eval_strategy": "steps",
    "remove_unused_columns": False,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "fp16": True,
}
if SELECTED_MAX_STEPS is not None:
    trainer_kwargs["max_steps"] = SELECTED_MAX_STEPS
else:
    trainer_kwargs["num_train_epochs"] = NUM_TRAIN_EPOCHS

FastLanguageModel.for_training(model)
data_collator = DataCollatorForLanguageModeling(tokenizer=base_tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    processing_class=base_tokenizer,
    data_collator=data_collator,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    args=TrainingArguments(**trainer_kwargs),
)

train_result = trainer.train()
print(train_result.metrics)


In [ ]:
import json
import platform

trainer.save_model(str(ADAPTER_DIR))
base_tokenizer.save_pretrained(str(ADAPTER_DIR))

local_run_spec = {
    "training_model": TRAINING_MODEL,
    "base_model": BASE_MODEL_METADATA,
    "output_dir": str(ADAPTER_DIR),
    "dataset_text_field": "text",
    "train_mode": TRAIN_MODE,
    "load_in_4bit": LOAD_IN_4BIT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": TRAIN_GRAD_ACCUM,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "warmup_steps": WARMUP_STEPS,
    "eval_steps": EVAL_STEPS,
    "save_steps": SAVE_STEPS,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "train_dataset_path": str(SELECTED_TRAIN_PATH),
    "eval_dataset_path": str(SELECTED_EVAL_PATH),
}
LOCAL_RUN_SPEC_PATH.write_text(json.dumps(local_run_spec, indent=2), encoding="utf-8")

manifest = {
    "output_run": OUTPUT_RUN,
    "dataset_repo": DATASET_REPO,
    "dataset_run": DATASET_RUN,
    "model_repo": MODEL_REPO,
    "adapter_dir": str(ADAPTER_DIR),
    "training_model": TRAINING_MODEL,
    "base_model": BASE_MODEL_METADATA,
    "train_mode": TRAIN_MODE,
    "gpu_name": gpu_name,
    "bf16_supported": bf16_supported,
    "platform": platform.platform(),
}
LOCAL_MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print({
    "adapter_dir": str(ADAPTER_DIR),
    "run_spec": str(LOCAL_RUN_SPEC_PATH),
    "manifest": str(LOCAL_MANIFEST_PATH),
})


In [ ]:
if UPLOAD_ADAPTER_TO_HF:
    model.push_to_hub_merged(MODEL_REPO, tokenizer, token=HF_TOKEN or None)

if EXPORT_GGUF:
    model.save_pretrained_gguf(str(GGUF_EXPORT_DIR), tokenizer, quantization_method="f16")
    print(f"GGUF export dir: {GGUF_EXPORT_DIR}")


## Notes

- This notebook is intentionally smoke-first for T4.
- If the smoke run still falls back to an obviously unusable path, keep the main AcidNet notebook for stronger GPUs and use this one only as a compatibility probe.
- If the smoke run looks healthy, you can re-run with `ACIDNET_TRAIN_MODE=full`.
- The restored AcidNet dataset still keeps the same `data/sft/`, `data/prompt_packs/`, `data/preferences/`, `data/training/`, and `data/eval/` layout used by the local project.
